In [ ]:
import re

import ndjson

with open(r"input.ndjson") as f:
    data = ndjson.load(f)

def extract_trip_id(trip_id):
    draft_prefix, trip_id = re.match(r"(drafts\.)?(.*)", trip_id).groups()
    is_draft = draft_prefix is not None
    return is_draft, trip_id

# Build map of trip ID to trip data
trip_map = {}
for row in data:
    if row["_type"] != "trip":
        continue
    is_draft, trip_id = extract_trip_id(row["_id"])
    if not is_draft and trip_id in trip_map:
        # Prefer draft version
        continue

    # Unpublish trip
    row["_id"] = "drafts." + trip_id

    # Update trip links to point to draft version
    report = row.get("report")
    if report is None:
        continue
    for block in report:
        if block["_type"] != "block":
            continue
        for markDef in block["markDefs"]:
            if markDef["_type"] != "tripLink":
                continue
            _, ref_trip_id = extract_trip_id(markDef["trip"]["_ref"])
            markDef["trip"]["_ref"] = "drafts." + ref_trip_id

    trip_map[trip_id] = row

# Extract non-trip rows
data = [row for row in data if row["_type"] != "trip"]

# Add updated trip rows
data.extend(trip_map.values())

with open("output.ndjson", "w") as f:
    ndjson.dump(data, f)